# Physics-Informed Neural Networks for Unsteady 2D Heat Transfer
## Simulating Meat Cooking on a Weber Grill
---
**Module:** Computational Fluid Dynamics  
**Department:** Mechanical & Aeronautical Engineering  
**Reference:** Raissi, M., Yazdani, A., & Karniadakis, G.E. (2020). *Hidden fluid mechanics.* Science, 367(6481), 1026–1030. https://doi.org/10.1126/science.aaw4741

---
### Learning Objectives
By the end of this notebook you will be able to:
1. Explain how a PINN differs from a standard data-driven ANN
2. Understand how the heat equation is embedded in the PINN loss via automatic differentiation
3. Train both models in PyTorch and interpret training and validation loss curves
4. Benchmark PINN and ANN predictions against FDM and an exact Fourier solution
5. Systematically investigate hyperparameter effects and explain the results
6. Evaluate model generalisation to an unseen material (lamb)

### Problem Statement
We solve the **2D unsteady heat conduction equation** on a rectangular domain
representing a cross-section of meat on a Weber grill (lid closed):

$$\\rho \\, c_p \\, \\frac{\\partial T}{\\partial t} = k \\left(\\frac{\\partial^2 T}{\\partial x^2} + \\frac{\\partial^2 T}{\\partial y^2}\\right)$$

| Quantity | Value |
|---------|-------|
| Domain | $x \\in [0,\\, 0.05\\,\\text{m}]$, $y \\in [0,\\, 0.15\\,\\text{m}]$, $t \\in [0,\\, 894\\,\\text{s}]$ |
| Initial condition | $T(x,y,0) = 288.15\\,\\text{K}$ (fridge temperature, ~15°C) |
| Boundary conditions | $T = 473.15\\,\\text{K}$ on all four walls (grill surface, ~200°C) |
| Material inputs | $\\rho$, $c_p$, $k$ — vary by meat type |

### Data Strategy
| Split | Contents | Purpose |
|-------|---------|---------|
| Train | 80% of combined beef/chicken/pork data | ANN weight updates |
| Validation | 20% of combined beef/chicken/pork data | Monitor overfitting |
| Test | Lamb (never seen during training) | Final generalisation evaluation |


In [ ]:
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    XMAX, YMAX, T_MAX, NX, NY, NT, DT,
    T_BOUNDARY, T_INITIAL, T_SAFE, DELTA_T,
    MEAT_PROPERTIES, TEST_MEAT,
)
from utils.dataset    import (build_ann_dataloaders, sample_collocation_points,
                               load_meat_data, normalise_inputs)
from utils.models     import ANN, PINN
from utils.training   import train_ann, train_pinn
from utils.evaluation import (evaluate_model, plot_loss_curves,
                               plot_field_comparison, plot_centre_temperature,
                               print_metrics_table)
from utils.fourier    import T_fourier_grid, T_fourier_centre_history
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")


---
## Section 2: Reference Data

In [ ]:
# ── Load FDM reference datasets ───────────────────────────────────────────────
DATA_DIR = "data"
fdm_data = {}

print("Loading FDM reference datasets...")
for meat in MEAT_PROPERTIES:
    raw   = load_meat_data(DATA_DIR, meat)
    props = MEAT_PROPERTIES[meat]
    nx_int, ny_int = NX - 2, NY - 2
    n_spatial = nx_int * ny_int

    x_grid = np.linspace(0, XMAX, NX)
    y_grid = np.linspace(0, YMAX, NY)
    t_grid = np.arange(NT) * DT

    u = np.full((NX, NY, NT), T_BOUNDARY)
    for it in range(NT):
        u[1:-1, 1:-1, it] = raw['T'][it*n_spatial:(it+1)*n_spatial].reshape(nx_int, ny_int)

    fdm_data[meat] = {'u': u, 'x': x_grid, 'y': y_grid, 't': t_grid,
                       'rho': props['rho'], 'cp': props['cp'], 'k': props['k']}
    alpha = props['k'] / (props['rho'] * props['cp'])
    print(f"  {meat:8s}: alpha={alpha:.3e} m²/s | T_centre(final)={u[NX//2,NY//2,-1]:.2f} K")


In [ ]:
# ── Visualise reference fields ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (meat, sol) in zip(axes, fdm_data.items()):
    im = ax.contourf(sol['y']*100, sol['x']*100, sol['u'][:,:,-1],
                     30, cmap='hot', vmin=T_INITIAL, vmax=T_BOUNDARY)
    plt.colorbar(im, ax=ax, label='T [K]')
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    ax.set_title(f"{meat.capitalize()} | α={alpha:.2e} m²/s", fontsize=10)
    ax.set_xlabel('y [cm]'); ax.set_ylabel('x [cm]')
plt.suptitle('FDM Reference — Final Temperature Field (t = 894 s)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Centre temperature vs time
plt.figure(figsize=(9, 5))
for meat, sol in fdm_data.items():
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    T_exact_c = T_fourier_centre_history(sol['t'], alpha)
    plt.plot(sol['t'], sol['u'][NX//2, NY//2, :], lw=2, label=f"{meat.capitalize()} FDM")
    plt.plot(sol['t'], T_exact_c, '--', lw=1.2, alpha=0.7, label=f"{meat.capitalize()} Exact")
plt.axhline(T_SAFE, color='red', ls=':', lw=1.5, label=f'Safe temp. {T_SAFE-273.15:.0f}°C', alpha=0.7)
plt.xlabel('Time [s]', fontsize=12); plt.ylabel('Centre Temperature [K]', fontsize=12)
plt.title('Centre Temperature — FDM vs Exact (Fourier)', fontsize=11)
plt.legend(fontsize=9, ncol=2); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## Section 3: Data Preparation

### 3.1 Input Normalisation
All six inputs are mapped to $[0, 1]$ before entering the network.

| Input | Normalisation |
|-------|--------------|
| $x$ | $\\hat{x} = x / L_x$ |
| $y$ | $\\hat{y} = y / L_y$ |
| $t$ | $\\hat{t} = t / t_{\\max}$ |
| $\\rho$, $c_p$, $k$ | min-max over training meat range |

### 3.2 Train / Validation Split
All three training meat datasets are combined into one pool, shuffled,
and split into training and validation subsets. The split ratio is
controlled by `VAL_SPLIT` in the hyperparameter block.

A validation loss that **diverges** from the training loss indicates overfitting.

### 3.3 Test Set
**Lamb** is never loaded during training or validation. It is held out
entirely and only used in Section 8 to evaluate generalisation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  HYPERPARAMETERS — modify these for experiments in Section 7
# ═══════════════════════════════════════════════════════════════════════════════

N_HIDDEN    = 5        # Number of hidden layers
N_NEURONS   = 40       # Neurons per hidden layer

LR          = 1e-3     # Adam learning rate
EPOCHS_ANN  = 5000     # Training epochs (ANN)
EPOCHS_PINN = 10000    # Training epochs (PINN)

LAMBDA_PDE  = 1.0      # PINN: weight on PDE residual loss
LAMBDA_BC   = 1.0      # PINN: weight on boundary condition loss
LAMBDA_IC   = 1.0      # PINN: weight on initial condition loss

N_PER_MEAT  = 5000     # Data points sampled per meat before split
VAL_SPLIT   = 0.2      # Fraction of combined data used for validation
BATCH_SIZE  = 512      # ANN mini-batch size

# ═══════════════════════════════════════════════════════════════════════════════

# ── ANN DataLoaders ───────────────────────────────────────────────────────────
print("Building ANN DataLoaders...")
train_loader, val_loader, n_train, n_val = build_ann_dataloaders(
    data_dir   = DATA_DIR,
    meat_names = list(MEAT_PROPERTIES.keys()),
    n_per_meat = N_PER_MEAT,
    val_split  = VAL_SPLIT,
    batch_size = BATCH_SIZE,
    device     = device,
    seed       = 42,
)

# ── PINN Collocation Points ───────────────────────────────────────────────────
print()
col, col_val = sample_collocation_points(
    N_col=8000, N_bc=600, N_ic=600,
    N_col_val=2000, N_bc_val=200, N_ic_val=200,
    seed=42,
)


---
## Section 4: Network Architectures

Both the ANN and PINN share **identical architecture**. The only difference
is the loss function — which is the central insight of this project.

| | ANN | PINN |
|--|-----|------|
| Architecture | Identical | Identical |
| Loss | MSE vs FDM data | PDE + BC + IC residuals |
| Training data | Required | Not required |
| Physics enforced | No | Yes (via autograd) |

### Why `tanh`?
The PINN computes second-order spatial derivatives of the network output
via automatic differentiation. The activation function must be at least
twice differentiable. `tanh` satisfies this. `ReLU` does not — its second
derivative is zero almost everywhere, making the PDE residual uninformative.

### PINN Loss

$$\\mathcal{L} = \\lambda_{\\text{pde}}\\,\\mathcal{L}_{\\text{pde}} + \\lambda_{\\text{bc}}\\,\\mathcal{L}_{\\text{bc}} + \\lambda_{\\text{ic}}\\,\\mathcal{L}_{\\text{ic}}$$

**PDE residual** at interior collocation points via autograd:

$$\\mathcal{L}_{\\text{pde}} = \\frac{1}{N_c}\\sum_{i=1}^{N_c}\\left[\\frac{\\rho c_p \\partial T/\\partial t - k(\\partial^2 T/\\partial x^2 + \\partial^2 T/\\partial y^2)}{\\rho c_p \\Delta T / t_{\\max}}\\right]^2$$

**Chain rule for normalised coordinates:**
$$\\frac{\\partial T}{\\partial t} = \\frac{\\partial T}{\\partial \\hat{t}} \\cdot \\frac{1}{t_{\\max}}, \\qquad \\frac{\\partial^2 T}{\\partial x^2} = \\frac{\\partial^2 T}{\\partial \\hat{x}^2} \\cdot \\frac{1}{L_x^2}$$


In [ ]:
ann  = ANN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS).to(device)
pinn = PINN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS,
             lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC,
             lambda_ic=LAMBDA_IC).to(device)

print("=" * 45)
ann.architecture_summary()
print()
pinn.architecture_summary()
print("=" * 45)


---
## Section 5: Training

In [ ]:
# ── Train ANN ─────────────────────────────────────────────────────────────────
loss_ann = train_ann(ann, train_loader, val_loader,
                      epochs=EPOCHS_ANN, lr=LR, device=device)


In [ ]:
# ── Train PINN ────────────────────────────────────────────────────────────────
loss_pinn = train_pinn(pinn, col, col_val,
                        epochs=EPOCHS_PINN, lr=LR, device=device)


In [ ]:
# ── Loss curves — training and validation ─────────────────────────────────────
plot_loss_curves(loss_ann, loss_pinn,
                 lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC, lambda_ic=LAMBDA_IC)


### Interpreting the Loss Curves

**ANN:** A gap between train and validation loss indicates overfitting —
the model has memorised the training data and does not generalise.
If the validation loss increases while training loss decreases, stop training earlier.

**PINN:** Train and validation losses use different (non-overlapping) collocation
points but the same physics. A large gap here is unusual — if it occurs, the
model may be fitting the specific random collocation points rather than the
underlying physics. Re-sampling collocation points or increasing `N_col` may help.


---
## Section 6: Evaluation and Benchmarking

In [ ]:
# ── Evaluate all training meats ───────────────────────────────────────────────
print("=" * 65)
eval_results = {}
for meat in MEAT_PROPERTIES:
    print()
    r_ann  = evaluate_model(ann,  meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    r_pinn = evaluate_model(pinn, meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    eval_results[meat] = {'ann': r_ann, 'pinn': r_pinn}
print("=" * 65)


In [ ]:
print_metrics_table(eval_results)

In [ ]:
for meat in MEAT_PROPERTIES:
    plot_field_comparison(meat, eval_results[meat]['ann'],
                           eval_results[meat]['pinn'],
                           fdm_data[meat]['x'], fdm_data[meat]['y'])


In [ ]:
for meat in MEAT_PROPERTIES:
    plot_centre_temperature(meat, MEAT_PROPERTIES[meat], fdm_data[meat],
                             models_dict={'ANN': ann, 'PINN': pinn}, device=device)


---
## Section 7: Hyperparameter Tuning  *(Student Task)*

### Your Task
Vary **one hyperparameter at a time**, keeping all others at baseline.  
For each experiment, record results in `experiment_log` and answer the
reflection questions in your report.

| Hyperparameter | Baseline | Suggested range |
|---------------|---------|----------------|
| `N_HIDDEN` | 5 | 2, 3, 5, 7, 9 |
| `N_NEURONS` | 40 | 10, 20, 40, 80, 128 |
| `LR` | 1e-3 | 1e-2, 1e-3, 5e-4, 1e-4 |
| `VAL_SPLIT` | 0.2 | 0.1, 0.2, 0.3 |
| `LAMBDA_PDE` | 1.0 | 0.1, 1.0, 5.0, 10.0 |
| `LAMBDA_BC` / `LAMBDA_IC` | 1.0 | 0.1, 1.0, 5.0, 10.0 |

### Reflection Questions

**Q1 — Depth vs width:** How does changing `N_HIDDEN` differ from changing
`N_NEURONS`? What does each affect about the network's capacity?

**Q2 — Overfitting:** For the ANN, at what point does the validation loss
diverge from the training loss? What does this mean physically?

**Q3 — Learning rate:** What happens at `LR = 1e-2`? At `LR = 1e-4`?
Why does Adam handle learning rate sensitivity better than SGD?

**Q4 — Loss weights (PINN):** Set `LAMBDA_BC = LAMBDA_IC = 0` and retrain.
What happens? What does this reveal about the role of BCs and ICs?

**Q5 — Activation (PINN):** Replace `tanh` with `ReLU`. What fails and why?

**Q6 — Generalisation:** Which model performs better on lamb (Section 8)?
Relate your answer to each model's loss function structure.


In [ ]:
import torch.nn as nn

def run_experiment(model_type='PINN',
                   n_hidden=5, n_neurons=40, lr=1e-3, epochs=5000,
                   val_split=0.2,
                   lambda_pde=1.0, lambda_bc=1.0, lambda_ic=1.0,
                   activation=None, label=None):
    """
    Train a fresh model with given hyperparameters and evaluate on beef.

    Parameters
    ----------
    model_type  : 'PINN' or 'ANN'
    n_hidden    : number of hidden layers
    n_neurons   : neurons per hidden layer
    lr          : Adam learning rate
    epochs      : training epochs
    val_split   : validation fraction (ANN only)
    lambda_*    : PINN loss weights (ignored for ANN)
    activation  : None (tanh default) or e.g. nn.ReLU(), nn.Sigmoid()
    label       : descriptive label for logging

    Returns
    -------
    model   : trained model
    history : loss history dict
    metrics : error metrics from evaluate_model()
    """
    if label is None:
        label = f"{model_type} | layers={n_hidden} | neurons={n_neurons} | lr={lr:.0e}"
    print(f"\nExperiment: {label}")
    print("-" * 60)

    if model_type == 'PINN':
        model = PINN(n_hidden=n_hidden, n_neurons=n_neurons,
                      lambda_pde=lambda_pde, lambda_bc=lambda_bc,
                      lambda_ic=lambda_ic).to(device)
    else:
        model = ANN(n_hidden=n_hidden, n_neurons=n_neurons).to(device)

    if activation is not None:
        for name, child in list(model.named_modules()):
            if isinstance(child, nn.Tanh):
                parts = name.split('.')
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1], activation)
        model._xavier_init()

    if model_type == 'PINN':
        history = train_pinn(model, col, col_val, epochs=epochs, lr=lr,
                              print_every=max(1, epochs//5), device=device)
    else:
        # Rebuild loaders with the requested val_split
        tr_l, vl_l, _, _ = build_ann_dataloaders(
            DATA_DIR, list(MEAT_PROPERTIES.keys()),
            n_per_meat=N_PER_MEAT, val_split=val_split,
            batch_size=BATCH_SIZE, device=device, seed=42)
        history = train_ann(model, tr_l, vl_l, epochs=epochs, lr=lr,
                             print_every=max(1, epochs//5), device=device)

    result = evaluate_model(model, 'beef', MEAT_PROPERTIES['beef'],
                             fdm_data['beef'], device=device)
    return model, history, result['metrics']


experiment_log = []

def log_experiment(label, metrics, notes=''):
    experiment_log.append({'label': label, **metrics, 'notes': notes})
    print(f"Logged: {label}")

def print_experiment_log():
    if not experiment_log:
        print("No experiments recorded yet.")
        return
    header = f"{'Label':<45} {'MAE (FDM)':>10} {'RMSE (FDM)':>11}  Notes"
    print(header); print("-" * len(header))
    for r in experiment_log:
        print(f"{r['label']:<45} {r.get('mae_fdm',float('nan')):>10.4f} "
              f"{r.get('rmse_fdm',float('nan')):>10.4f}   {r.get('notes','')}")

# ── Example (uncomment to run) ────────────────────────────────────────────────
# model_exp1, hist_exp1, met_exp1 = run_experiment(
#     model_type='PINN', n_hidden=3, n_neurons=20, epochs=5000,
#     label='Shallow PINN: 3 layers x 20 neurons')
# log_experiment('Shallow PINN 3x20', met_exp1, notes='Under-parameterised')

print("Experiment runner ready.")


In [ ]:
print_experiment_log()

---
## Section 8: Test Meat — Generalisation Evaluation  *(Student Task)*

Your models were trained on **beef, chicken, and pork**.  
Now evaluate your best models on **lamb** — never seen during training.

### Your Task
1. Replace `best_ann` and `best_pinn` with your best models from Section 7
2. Evaluate on lamb and compare errors to the training meat results
3. In your report, explain which model generalises better and why


In [ ]:
# ── Load lamb FDM reference ───────────────────────────────────────────────────
print("Loading test meat (lamb) reference...")
raw_lamb   = load_meat_data(DATA_DIR, 'lamb')
props_lamb = TEST_MEAT['lamb']
nx_int, ny_int = NX - 2, NY - 2
n_spatial = nx_int * ny_int

x_grid = np.linspace(0, XMAX, NX)
y_grid = np.linspace(0, YMAX, NY)
t_grid = np.arange(NT) * DT

u_lamb = np.full((NX, NY, NT), T_BOUNDARY)
for it in range(NT):
    u_lamb[1:-1, 1:-1, it] = raw_lamb['T'][it*n_spatial:(it+1)*n_spatial].reshape(nx_int, ny_int)

fdm_lamb = {'u': u_lamb, 'x': x_grid, 'y': y_grid, 't': t_grid, **props_lamb}
alpha_lamb = props_lamb['k'] / (props_lamb['rho'] * props_lamb['cp'])
print(f"Lamb: alpha={alpha_lamb:.3e} m²/s | T_centre(final)={u_lamb[NX//2,NY//2,-1]:.2f} K")


In [ ]:
# Replace with your best models from Section 7
best_ann  = ann
best_pinn = pinn

print("ANN on lamb:")
r_ann_lamb  = evaluate_model(best_ann,  'lamb', props_lamb, fdm_lamb, device=device)
print()
print("PINN on lamb:")
r_pinn_lamb = evaluate_model(best_pinn, 'lamb', props_lamb, fdm_lamb, device=device)


In [ ]:
plot_field_comparison('lamb', r_ann_lamb, r_pinn_lamb, x_grid, y_grid)

In [ ]:
plot_centre_temperature('lamb', props_lamb, fdm_lamb,
                         models_dict={'ANN': best_ann, 'PINN': best_pinn},
                         device=device)


---
## Section 9: Report Guidance

### 1. Model Understanding
- Describe the network architecture and the role of each hyperparameter
- Derive the chain rule used to compute physical derivatives from normalised-coordinate outputs
- Explain the composite PINN loss — what does each term enforce?
- Why is `tanh` required for PINN autograd?

### 2. Hyperparameter Study
- Present all experiments using `print_experiment_log()`
- For each hyperparameter: describe the trend and provide a physical/mathematical explanation
- Include train vs validation loss curves for each experiment — discuss any overfitting

### 3. Benchmarking — Training Meats
- Compare ANN, PINN, FDM, and Exact (MAE, RMSE) for all three training meats
- Plot temperature fields and centre temperature histories
- Discuss where each method performs best and worst

### 4. Generalisation — Test Meat (Lamb)
- Compare ANN and PINN on the unseen test material
- Which generalises better, and why?

### 5. Critical Method Comparison

| Criterion | FDM | ANN | PINN |
|-----------|-----|-----|------|
| Requires simulation data | ✓ | ✓ | ✗ |
| Enforces physics | ✓ | ✗ | ✓ |
| Generalises across materials | Re-run required | Partially | ✓ |
| Inference cost | High | Low | Low |
| Interpretability | High | Low | Medium |

---
*Department of Mechanical & Aeronautical Engineering | University of Pretoria*
